# 🧬 DeepSeek-V4 架构总览 — 从 V3 到 V4 的技术跃迁

**来源**: [LMSys Blog: DeepSeek-V4 Technical Deep Dive](https://www.lmsys.org/blog/2026-04-25-deepseek-v4/) (2026.04)

**本文目标**：建立 DeepSeek-V4 的完整架构心智模型，理解从 V3 到 V4 的核心技术创新。

读完这篇你会理解：
- V4 两个变体 (Pro 1.6T / Flash 284B) 的定位差异
- 五大核心创新: Hybrid Sparse Attention / mHC / FP4 权重 / ShadowRadix / HiSparse
- 与 V3 的关键架构变化
- 100 万 token 上下文是如何实现的

## 1. 模型谱系: V3 → V3.2 → V4

```
DeepSeek 系列的架构演进:

V3 (2024.12):
  ├─ MoE 架构 (256 experts, top-8 routing)
  ├─ Multi-head Latent Attention (MLA)
  ├─ Auxiliary-loss-free load balancing
  └─ FP8 训练

V3.1 (2025):
  ├─ 增强的 RL 训练 (DAPO/GRPO)
  └─ 扩展的 reasoning 能力

V3.2 (2025):
  ├─ DeepSeek Sparse Attention (DSA)  ← 首次引入稀疏注意力
  ├─ 改进的 MoE routing
  └─ 128K context window

V4 (2026.04):
  ├─ Hybrid Sparse Attention (SWA + C4/C128)
  ├─ mHC (Manifold-Constrained Hyper-Connections) 替代残差连接
  ├─ FP4 原生专家权重 (Blackwell 优化)
  ├─ 1M token 上下文窗口
  ├─ ShadowRadix (原生前缀缓存)
  └─ HiSparse (分层内存稀疏注意力)
```

### 1.1 两个变体

| | DeepSeek-V4 Pro | DeepSeek-V4 Flash |
|---|---|---|
| 参数总量 | 1.6T (1600B) | 284B |
| 激活参数 | ~36B (估算) | ~18B (估算) |
| 定位 | 最强性能 (旗舰) | 高效推理 (性价比) |
| 部署硬件 | B200 x8 (TP8) | H200 x4 (TP4) 或 B200 x2 |
| 上下文 | 1M tokens | 1M tokens |

## 2. 五大核心创新一览

```
DeepSeek-V4 = MoE Backbone
            + Hybrid Sparse Attention  (替代 Dense/DSA)
            + mHC Connections          (替代 Residual)
            + FP4 Expert Weights       (替代 FP8/BF16)
            + ShadowRadix Cache        (推理: 前缀缓存)
            + HiSparse Memory          (推理: 分层显存)
```

### 2.1 Hybrid Sparse Attention — "每一层都在 SWA 和压缩注意力之间选择"

传统 Transformer: 每层都做 Full Self-Attention → O(n²)

V4 的方案:
- **每一层**在两种配置中选择其一:
  - **SWA + C4**: 滑动窗口 (128 tokens) + 4:1 压缩后 top-512 稀疏注意力
  - **SWA + C128**: 滑动窗口 (128 tokens) + 128:1 压缩后全量密集注意力

- SWA 保证了"局部上下文"不丢失 (最近 128 tokens 的精细注意力)
- C4/C128 负责"全局上下文" (通过压缩实现覆盖 1M tokens)

### 2.2 mHC — "不只是加残差，而是学习的混合权重"

标准残差: `output = x + f(x)`

mHC: `output = mix(x₁, x₂, ..., x_hc_mult)`
- 在 `hc_mult` 个并行分支上运行
- 用 Sinkhorn 归一化产生 per-token 的混合权重
- 改善了梯度流和表示质量

### 2.3 FP4 专家权重 — "推理速度翻倍的秘诀"

- MoE 的专家权重以**原生 FP4** 精度存储
- 专为 Blackwell 架构的 FP4 Tensor Core 设计
- FP8×FP4 GEMM → 比 FP8×FP8 快 ~2x
- 训练时 BF16/FP8, 推理时 FP4

### 2.4 ShadowRadix — "为混合注意力设计的原生前缀缓存"

- 扩展了 SGLang 的 RadixAttention 思想
- 核心: 一个虚拟的全 token slot radix tree 作为"统一坐标系"
- 每个物理池 (SWA/C4/C128) 从中"投影"出索引映射 (影子)
- 解决了混合注意力下前缀缓存的复杂性

### 2.5 HiSparse — "把不活跃的 KV Cache 换出到 CPU"

- 针对 C4 层: 每步只有少量压缩位置被 indexer 的 top-k 选中
- 大部分 C4 KV 处于"不活跃"状态 → 可以放在 CPU 内存
- 长上下文推理的 token 容量和吞吐提升最高 3x

## 3. 模型架构全景图

```
┌─────────────────────────────────────────────────────────────────┐
│                    DeepSeek-V4 模型架构                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│  Input Tokens                                                     │
│      │                                                            │
│  ┌───┴──────────────────────────────────────────────────────┐    │
│  │              Embedding + mHC Init                          │    │
│  └───┬──────────────────────────────────────────────────────┘    │
│      │                                                            │
│  ┌───┴──────────────────────────────────────────────────────┐    │
│  │         Transformer Layers (N layers)                      │    │
│  │                                                            │    │
│  │  每层包含:                                                  │    │
│  │  ┌──────────────────────────────────────────────────┐     │    │
│  │  │  mHC (替代 Residual)                              │     │    │
│  │  │  ├─ SWA (滑动窗口, 128 tokens)                    │     │    │
│  │  │  ├─ C4 (4:1 压缩, top-512)  ← 或 C128            │     │    │
│  │  │  └─ FlashMLA (融合内核)                           │     │    │
│  │  ├──────────────────────────────────────────────────┤     │    │
│  │  │  MoE FFN (DeepEP + DeepGEMM Mega MoE)            │     │    │
│  │  │  ├─ Router (top-k expert selection)              │     │    │
│  │  │  ├─ FP4 Expert Weights                          │     │    │
│  │  │  └─ Shared Expert (dense)                        │     │    │
│  │  └──────────────────────────────────────────────────┘     │    │
│  │                                                            │    │
│  └───┬──────────────────────────────────────────────────────┘    │
│      │                                                            │
│  ┌───┴──────────────────────────────────────────────────────┐    │
│  │              mHC Output + LM Head                          │    │
│  └──────────────────────────────────────────────────────────┘    │
│                                                                   │
└─────────────────────────────────────────────────────────────────┘

Memory Layout (推理时, B200, TP=8):
┌──────────┬──────────────┬───────────────┬────────────┬──────────┐
│ Weights  │ SWA KV Cache │ C4/C128 Cache │ Act/Misc   │ HiSparse │
│ (FP4 MoE)│ (128 tokens) │ (compressed)   │            │ CPU pool │
│ ~200 GB  │ ~1 GB        │ ~20 GB         │ ~5 GB      │ ~100 GB  │
│ (TP分片) │              │                │            │ (CPU)    │
└──────────┴──────────────┴───────────────┴────────────┴──────────┘
```

## 4. 与 V3 的关键架构对比

| 维度 | V3/V3.2 | V4 |
|------|---------|-----|
| Attention | MLA (Dense) / DSA (Sparse) | **Hybrid Sparse (SWA+C4/C128)** |
| 残差连接 | Standard Residual | **mHC (Sinkhorn-weighted)** |
| 专家权重精度 | FP8/BF16 | **原生 FP4** |
| 最大上下文 | 128K | **1M** |
| 前缀缓存 | 外部 (SGLang) | **ShadowRadix (原生)** |
| KV Cache 卸载 | 无 | **HiSparse (CPU offload)** |
| 投机解码 | MTP (多 token 预测) | **单层 MTP (仅 SWA)** |
| 训练并行策略 | DP/TP/EP/PP | **DP/TP/SP/EP/PP/CP (6种)** |

## 5. 阅读路线

建议按以下顺序深入:
1. **Hybrid Sparse Attention 深度解析** — SWA/C4/C128 的工作原理
2. **ShadowRadix 前缀缓存** — 虚拟 token 槽 + 影子投影
3. **HiSparse 分层内存** — CPU offloading 机制
4. **训练: Miles + Megatron-LM** — 6 种并行策略 + RL 训练
5. **推理优化与部署** — 投机解码 + PD 分离 + CUDA graph